# SecureBox - Demo del canal seguro

Este notebook muestra paso a paso cómo funciona el protocolo de handshake de SecureBox entre dos partes: **Alice** y **Bob**.

El protocolo sigue este flujo:
1. Cada parte genera un par de claves X25519 efímeras y las intercambia.
2. Ambas partes calculan el mismo secreto compartido y derivan dos claves AES-256 con HKDF.
3. Cada parte firma el transcript del handshake con Ed25519 para evitar ataques MITM.
4. Los mensajes se cifran con AES-256-GCM usando un contador como nonce.

## Imports

In [14]:
from cryptography.hazmat.primitives.asymmetric import x25519, ed25519
from cryptography.hazmat.primitives.serialization import Encoding, PublicFormat
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.exceptions import InvalidSignature

## Definición de funciones

Definimos todas las funciones necesarias para el protocolo.

In [15]:
# Contextos HKDF para separar direcciones
HKDF_INFO_ALICE_TO_BOB = b"securebox-v1-handshake-alice-to-bob"
HKDF_INFO_BOB_TO_ALICE = b"securebox-v1-handshake-bob-to-alice"


def generar_efimera():
    """Genera un par de claves X25519 efímeras. Devuelve (priv, pub_bytes)."""
    priv = x25519.X25519PrivateKey.generate()
    pub_bytes = priv.public_key().public_bytes(encoding=Encoding.Raw, format=PublicFormat.Raw)
    return priv, pub_bytes


def derivar_claves(mi_priv, mi_pub_bytes, peer_pub_bytes):
    """Calcula el secreto compartido y deriva dos claves AES-256 con HKDF. Devuelve (send_key, recv_key)."""
    peer_pub = x25519.X25519PublicKey.from_public_bytes(peer_pub_bytes)
    shared_secret = mi_priv.exchange(peer_pub)

    # Transcript determinista: menor || mayor
    if mi_pub_bytes < peer_pub_bytes:
        transcript = mi_pub_bytes + peer_pub_bytes
    else:
        transcript = peer_pub_bytes + mi_pub_bytes

    key_a2b = HKDF(algorithm=hashes.SHA256(), length=32, salt=transcript, info=HKDF_INFO_ALICE_TO_BOB).derive(shared_secret)
    key_b2a = HKDF(algorithm=hashes.SHA256(), length=32, salt=transcript, info=HKDF_INFO_BOB_TO_ALICE).derive(shared_secret)

    # Quien tiene pub menor es Alice: envía con a2b, recibe con b2a
    if mi_pub_bytes < peer_pub_bytes:
        return key_a2b, key_b2a
    else:
        return key_b2a, key_a2b


def firmar_transcript(mi_pub_bytes, peer_pub_bytes, sign_priv):
    """Firma el transcript del handshake con Ed25519. Devuelve la firma."""
    if mi_pub_bytes < peer_pub_bytes:
        transcript = mi_pub_bytes + peer_pub_bytes
    else:
        transcript = peer_pub_bytes + mi_pub_bytes
    return sign_priv.sign(transcript)


def verificar_transcript(mi_pub_bytes, peer_pub_bytes, peer_sign_pub, firma):
    """Verifica la firma del transcript del otro lado. Lanza ValueError si falla."""
    if mi_pub_bytes < peer_pub_bytes:
        transcript = mi_pub_bytes + peer_pub_bytes
    else:
        transcript = peer_pub_bytes + mi_pub_bytes
    try:
        peer_sign_pub.verify(firma, transcript)
    except InvalidSignature:
        raise ValueError("Firma del transcript inválida. Posible ataque MITM.")


def enviar(send_key, counter, mensaje):
    """Cifra un mensaje con AES-256-GCM. El nonce se construye a partir del contador."""
    if isinstance(mensaje, str):
        mensaje = mensaje.encode("utf-8")
    nonce = counter.to_bytes(12, byteorder="big")
    ciphertext = AESGCM(send_key).encrypt(nonce, mensaje, associated_data=None)
    return {"counter": counter, "ciphertext": ciphertext}


def recibir(recv_key, counter_esperado, paquete):
    """Descifra y verifica un mensaje. Detecta replays y modificaciones."""
    if paquete["counter"] != counter_esperado:
        raise ValueError(
            f"Replay o mensaje fuera de orden. "
            f"Esperado: {counter_esperado}, recibido: {paquete['counter']}."
        )
    nonce = counter_esperado.to_bytes(12, byteorder="big")
    try:
        plaintext = AESGCM(recv_key).decrypt(nonce, paquete["ciphertext"], associated_data=None)
    except Exception:
        raise ValueError("Fallo de autenticación: el mensaje fue modificado en tránsito.")
    return plaintext.decode("utf-8")


def gen_sign_keypair():
    """Genera un par de claves Ed25519 para firmas digitales."""
    priv = ed25519.Ed25519PrivateKey.generate()
    return priv, priv.public_key()


print('Funciones definidas correctamente.')

Funciones definidas correctamente.


## Setup - Claves de identidad

Antes de empezar el handshake, cada parte tiene un par de claves Ed25519 de identidad (long-term). Estas claves son permanentes y se usan para autenticar el transcript, no para cifrar mensajes.

In [16]:
alice_sign_priv, alice_sign_pub = gen_sign_keypair()
bob_sign_priv,   bob_sign_pub   = gen_sign_keypair()

print('Claves de identidad Ed25519 generadas para Alice y Bob.')

Claves de identidad Ed25519 generadas para Alice y Bob.


## Paso 1 - Intercambio de claves efímeras X25519

Cada parte genera un par de claves X25519 efímeras, es decir, exclusivas para esta sesión. Las claves privadas nunca se comparten. Solo se intercambian las claves públicas.

Usar claves efímeras garantiza forward secrecy: aunque la clave de identidad se comprometa en el futuro, los mensajes de sesiones pasadas no pueden descifrarse.

In [17]:
alice_priv, alice_pub = generar_efimera()
bob_priv,   bob_pub   = generar_efimera()

print(f'Alice pub: {alice_pub.hex()[:40]}...')
print(f'Bob pub: {bob_pub.hex()[:40]}...')

Alice pub: 1047d137200c1b7c97e6bd64978146d1c77c66e2...
Bob pub: 8cbea5c05641310e7e816059e662d0d6a91d16e5...


## Paso 2 - Derivación de claves con HKDF

Ambas partes calculan el mismo secreto compartido mediante ECDH: Alice usa su clave privada efímera con la pública de Bob, y viceversa. El resultado es idéntico en ambos lados sin haber transmitido el secreto.

A partir del secreto compartido se derivan dos claves AES-256 independientes con HKDF-SHA256, una para cada dirección del canal (`alice->bob` y `bob->alice`). Usar claves distintas por dirección evita ataques de reflexión.

El transcript (concatenación ordenada de las dos claves efímeras públicas) se usa como salt en HKDF, vinculando las claves derivadas al intercambio concreto.

In [18]:
alice_send, alice_recv = derivar_claves(alice_priv, alice_pub, bob_pub)
bob_send,   bob_recv   = derivar_claves(bob_priv,   bob_pub,   alice_pub)

print(f'Alice send key: {alice_send.hex()}')
print(f'Bob recv key: {bob_recv.hex()}')
print(f'Claves iguales: {alice_send == bob_recv}')

Alice send key: 1d0be45f4bae06c59bd8f7c90218426a4729f347cd2921e182245f5e9124ddad
Bob recv key: 1d0be45f4bae06c59bd8f7c90218426a4729f347cd2921e182245f5e9124ddad
Claves iguales: True


## Paso 3 - Autenticación del transcript con Ed25519

Para evitar ataques MITM (man-in-the-middle), cada parte firma el transcript del handshake con su clave de identidad Ed25519 y verifica la firma de la otra parte.

El transcript es la concatenación ordenada de las dos claves efímeras públicas (de menor a mayor en bytes), garantizando que ambas partes construyan exactamente el mismo valor.

Si un atacante hubiera sustituido alguna clave efímera en tránsito, la verificación fallaría porque la firma no correspondería al transcript manipulado.

In [19]:
alice_sig = firmar_transcript(alice_pub, bob_pub, alice_sign_priv)
bob_sig   = firmar_transcript(bob_pub,   alice_pub, bob_sign_priv)

verificar_transcript(alice_pub, bob_pub, bob_sign_pub,   bob_sig)
verificar_transcript(bob_pub,   alice_pub, alice_sign_pub, alice_sig)

print('Transcript firmado y verificado por ambas partes. Sin MITM.')

Transcript firmado y verificado por ambas partes. Sin MITM.


## Paso 4 - Intercambio de mensajes cifrados con AES-256-GCM

Una vez completado el handshake, los mensajes se cifran con AES-256-GCM. El nonce de cada mensaje se construye a partir del contador de mensajes enviados, codificado en 12 bytes en big-endian. Esto garantiza que cada mensaje use un nonce distinto sin necesidad de almacenar nonces previos.

El receptor verifica que el contador del paquete recibido coincide con el esperado, detectando así cualquier intento de replay.

In [20]:
contadores = {'alice_sc': 0, 'bob_sc': 0, 'alice_rc': 0, 'bob_rc': 0}

conversacion = [
    ('Alice', alice_send, bob_recv,   'alice_sc', 'bob_rc',   'Hola Bob, ¿me recibes?'),
    ('Bob',   bob_send,   alice_recv, 'bob_sc',   'alice_rc', 'Sí Alice, canal seguro establecido.'),
    ('Alice', alice_send, bob_recv,   'alice_sc', 'bob_rc',   'Perfecto. Aquí va un secreto: clave=42.'),
    ('Bob',   bob_send,   alice_recv, 'bob_sc',   'alice_rc', 'Recibido. Guardado de forma segura.'),
    ('Alice', alice_send, bob_recv,   'alice_sc', 'bob_rc',   'Cerrando canal. Hasta pronto.'),
]

for sender_name, send_key, recv_key, sc_key, rc_key, texto in conversacion:
    paquete = enviar(send_key, contadores[sc_key], texto)
    recibido = recibir(recv_key, contadores[rc_key], paquete)
    receptor = 'Bob' if sender_name == 'Alice' else 'Alice'
    print(f'{sender_name} -> [{paquete["counter"]}] cifrado: {paquete["ciphertext"].hex()[:20]}...')
    print(f'{receptor} descifra: "{recibido}"')
    print()
    contadores[sc_key] += 1
    contadores[rc_key] += 1

Alice -> [0] cifrado: 580aa93f76ae3a9c74ab...
Bob descifra: "Hola Bob, ¿me recibes?"

Bob -> [0] cifrado: 5ce3bd436d9a08f716fb...
Alice descifra: "Sí Alice, canal seguro establecido."

Alice -> [1] cifrado: ad960d1f913f403b32ae...
Bob descifra: "Perfecto. Aquí va un secreto: clave=42."

Bob -> [1] cifrado: 3a800407f8ffcb99e4db...
Alice descifra: "Recibido. Guardado de forma segura."

Alice -> [2] cifrado: 60392eebfc7e48dc2e6b...
Bob descifra: "Cerrando canal. Hasta pronto."



## Prueba de seguridad 1 - Modificación de mensaje en tránsito

Si el ciphertext de un mensaje es alterado en tránsito, el tag de autenticación de AES-256-GCM no coincidirá y el descifrado lanzará una excepción.

In [21]:
paquete = enviar(alice_send, contadores['alice_sc'], 'Mensaje que será modificado')
paquete['ciphertext'] = b'\x00' * len(paquete['ciphertext'])

try:
    recibir(bob_recv, contadores['bob_rc'], paquete)
    print('ERROR: no se detectó la modificación')
except ValueError as e:
    print(f'Modificación detectada correctamente: {e}')

Modificación detectada correctamente: Fallo de autenticación: el mensaje fue modificado en tránsito.


## Prueba de seguridad 2 - Replay de un mensaje anterior

Si un atacante reenvía un paquete ya recibido, el contador no coincidirá con el esperado y el mensaje será rechazado.

In [23]:
priv2, pub2 = generar_efimera()
priv3, pub3 = generar_efimera()
send2, _    = derivar_claves(priv2, pub2, pub3)
_, recv3    = derivar_claves(priv3, pub3, pub2)

paquete_original = enviar(send2, 0, 'Mensaje normal')
recibir(recv3, 0, paquete_original)

try:
    recibir(recv3, 1, paquete_original)
    print('ERROR: no se detectó el replay')
except ValueError as e:
    print(f'Replay detectado correctamente: {e}')

Replay detectado correctamente: Replay o mensaje fuera de orden. Esperado: 1, recibido: 0.
